# BACH (ICIAR-2018) Downloader & Setup for GPCN-ViT

Downloads the **BACH breast-cancer histology** dataset from **Kaggle** and arranges it into the
exact folder layout the GPCN-ViT project expects, so `bach.py` (CELL 13 of the main
`GPCN_ViT_Colab.ipynb`) can train on it.

**Source (authentic):** Kaggle mirror `truthisneverlinear/bach-breast-cancer-histology-images`
of the official ICIAR-2018 BACH challenge data (Araújo et al., *Med. Image Anal.* 2019,
DOI 10.1016/j.media.2019.05.010). License CC BY-NC-ND — research use, cite the paper.

**What this notebook produces:**
```
/content/drive/MyDrive/MY RESEARCH/BACH/Photos/
    Normal/    Benign/    InSitu/    Invasive/      <- *.tif images
```
which is exactly the `BACH_PATH` the main notebook's CELL 2 points at.

**You need a Kaggle API token** (`kaggle.json`): Kaggle → your avatar → *Settings* →
*API* → **Create New Token**. CELL 2 will ask you to upload it.

> The 4 microscopy classes are kept; the BACH whole-slide images (WSI, a different
> format/task) are ignored. `bach.py` pools whatever class images it finds and
> auto-detects the number of classes, so the exact per-class counts of the mirror
> do not need to match the canonical 100/class.

In [ ]:
# ── CELL 1 ── Install Kaggle CLI + mount Google Drive
!pip -q install kaggle

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted.')

In [ ]:
# ── CELL 2 ── Kaggle credentials
# Option A (default): upload kaggle.json  (Kaggle > Settings > API > Create New Token)
import os
from pathlib import Path
from google.colab import files

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
token = kaggle_dir / 'kaggle.json'

if not token.exists():
    print('Upload your kaggle.json:')
    up = files.upload()                     # choose your kaggle.json file
    for name, data in up.items():
        if name.endswith('.json'):
            token.write_bytes(data)
os.chmod(token, 0o600)
print('kaggle.json ready at', token)

# Option B (alternative): use Colab Secrets instead of uploading a file
# from google.colab import userdata
# os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
# os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

In [ ]:
# ── CELL 3 ── Download + unzip the BACH dataset
KAGGLE_SLUG  = 'truthisneverlinear/bach-breast-cancer-histology-images'
DOWNLOAD_DIR = '/content/bach_raw'

!rm -rf {DOWNLOAD_DIR} && mkdir -p {DOWNLOAD_DIR}
!kaggle datasets download -d {KAGGLE_SLUG} -p {DOWNLOAD_DIR} --unzip

print('\nExtracted directory tree (top 3 levels):')
!find {DOWNLOAD_DIR} -maxdepth 3 -type d | sort | head -60

In [ ]:
# ── CELL 4 ── Consolidate into the project layout: BACH/Photos/<Class>
# Robust: finds class folders ANYWHERE in the extracted tree (handles Train/Test
# splits, Photos/ wrappers, nested dirs) using the SAME class-name normalisation
# as bach.py, and pools everything into Photos/<Class>. Idempotent + collision-safe.
import shutil
from pathlib import Path
from collections import defaultdict

DOWNLOAD_DIR = '/content/bach_raw'
BACH_ROOT    = '/content/drive/MyDrive/MY RESEARCH/BACH'   # parent of Photos/ (match main notebook)
PHOTOS_DIR   = Path(BACH_ROOT) / 'Photos'

CANON   = ['normal', 'benign', 'insitu', 'invasive']
DISPLAY = {'normal': 'Normal', 'benign': 'Benign', 'insitu': 'InSitu', 'invasive': 'Invasive'}
IMG_EXT = {'.tif', '.tiff', '.png', '.jpg', '.jpeg', '.bmp'}

def _norm(s):
    return s.lower().replace(' ', '').replace('_', '').replace('-', '').strip()

def class_of(path):
    # nearest ancestor folder whose normalised name matches a canonical class
    for part in reversed(path.parts[:-1]):
        k = _norm(part)
        for c in CANON:
            if c in k:
                return c
    return None

PHOTOS_DIR.mkdir(parents=True, exist_ok=True)
for c in CANON:
    (PHOTOS_DIR / DISPLAY[c]).mkdir(exist_ok=True)

counts = defaultdict(int); copied = skipped = 0
for p in Path(DOWNLOAD_DIR).rglob('*'):
    if not p.is_file() or p.suffix.lower() not in IMG_EXT:
        continue
    c = class_of(p)
    if c is None:                 # whole-slide / unlabeled -> skip
        skipped += 1
        continue
    dst = PHOTOS_DIR / DISPLAY[c] / p.name
    if dst.exists():              # avoid clobbering same-named files (e.g. Train/ vs Test/)
        dst = PHOTOS_DIR / DISPLAY[c] / f'{p.parent.name}_{p.name}'
    if not dst.exists():
        shutil.copy2(p, dst); copied += 1
    counts[c] += 1

print(f'Consolidated into: {PHOTOS_DIR}')
print(f'  copied {copied} new files | skipped {skipped} non-class/WSI files\n')
total = 0
for c in CANON:
    n = counts[c]; total += n
    print(f'  {DISPLAY[c]:<9}: {n}' + ('' if n else '   [!] EMPTY'))
print(f'  {"TOTAL":<9}: {total}')

if total == 0:
    print('\n[!] No class images found. Inspect CELL 3 tree and check folder names.')
else:
    print('\nNext: in GPCN_ViT_Colab.ipynb CELL 2 make sure you have')
    print(f"      BACH_PATH = '{PHOTOS_DIR}'")
    print('then run that notebook\'s CELL 13 (train_bach) to train on BACH.')

In [ ]:
# ── CELL 5 ── (optional) Verify with the project's own loader
# Run the MAIN notebook's CELL 3 first (it copies bach.py to /content), or skip.
import importlib.util
PHOTOS_DIR = '/content/drive/MyDrive/MY RESEARCH/BACH/Photos'
if importlib.util.find_spec('bach') is not None:
    from bach import build_bach_index
    index, class_names = build_bach_index(PHOTOS_DIR)
    print(f'OK — bach.build_bach_index found {len(index)} images, classes = {class_names}')
else:
    print('bach.py not on the Python path yet.')
    print('Copy the project files first (main notebook CELL 3), then re-run this cell.')